# Martingale Staking — Drawdown & Streak Analysis

Simulates two-team sports markets and evaluates a **Martingale recoup staking system** under two strategies:

- **Favourites agent** — always bets on the team with lower odds (higher implied probability)
- **Outsiders agent** — always bets on the team with higher odds (lower implied probability)

### Staking rule
- First bet (and any bet after a win): **\$1 stake**
- After a loss: stake enough to **recoup all accumulated losses + \$1 profit** at the current market's odds
  - `stake = (cumulative_losses + 1) / ((odds - 1) × (1 - commission))`
- A win resets the stake sequence back to \$1

### Key questions
1. What drawdowns should you expect, and how bad can they get?
2. How long are typical losing streaks, and what is the worst-case?
3. How much starting bankroll do you actually need?
4. Does betting favourites vs. outsiders change the risk profile meaningfully?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.gridspec import GridSpec

rng = np.random.default_rng(seed=42)

# ── Simulation parameters ──────────────────────────────────────────────────
N_BETS          = 10_000    # bets per simulation
N_SIMS          = 1_000     # Monte Carlo runs
STARTING_BAL    = 10_000.0  # starting bankroll ($)
OVERROUND       = 1.05      # 105% book
COMMISSION      = 0.05      # 5% commission on winning profit
MIN_TRUE_PROB   = 0.30      # narrowest favourite
MAX_TRUE_PROB   = 0.70      # widest favourite

FAVOURITE_COLOR = '#2196F3'
OUTSIDER_COLOR  = '#FF5722'

## Simulation engine

In [ ]:
def simulate_martingale(
    agent_type: str,
    n_bets: int = N_BETS,
    starting_balance: float = STARTING_BAL,
    overround: float = OVERROUND,
    commission: float = COMMISSION,
    min_true_prob: float = MIN_TRUE_PROB,
    max_true_prob: float = MAX_TRUE_PROB,
    local_rng=None,
) -> dict:
    """
    Run one Martingale simulation.

    Parameters
    ----------
    agent_type : 'favourite' or 'outsider'

    Returns
    -------
    dict with:
        final_balance, max_drawdown, max_win_streak, max_loss_streak,
        max_stake, bust (bool), bust_step, balance_history (np.ndarray)
    """
    if local_rng is None:
        local_rng = rng

    # Pre-generate all markets for this simulation
    true_prob_a = local_rng.uniform(min_true_prob, max_true_prob, size=n_bets)
    odds_a = 1.0 / (true_prob_a * overround)
    odds_b = 1.0 / ((1.0 - true_prob_a) * overround)
    team_a_wins = local_rng.random(size=n_bets) < true_prob_a

    balance          = starting_balance
    cumulative_loss  = 0.0
    peak_balance     = starting_balance
    max_drawdown     = 0.0
    max_stake        = 0.0
    cur_win_streak   = 0
    cur_loss_streak  = 0
    max_win_streak   = 0
    max_loss_streak  = 0
    bust             = False
    bust_step        = None

    balance_hist = np.empty(n_bets + 1)
    balance_hist[0] = balance

    steps_completed = 0

    for i in range(n_bets):
        # ── Pick team ──────────────────────────────────────────────────
        if agent_type == 'favourite':
            bet_on_a = odds_a[i] <= odds_b[i]   # shorter odds = favourite
        else:
            bet_on_a = odds_a[i] > odds_b[i]    # longer odds = outsider

        bet_odds = odds_a[i] if bet_on_a else odds_b[i]

        # ── Martingale stake ───────────────────────────────────────────
        if cumulative_loss == 0.0:
            stake = 1.0
        else:
            net_profit_factor = (bet_odds - 1.0) * (1.0 - commission)
            stake = (cumulative_loss + 1.0) / net_profit_factor

        # ── Bust check: can't cover the required stake ─────────────────
        if stake > balance:
            bust      = True
            bust_step = i
            break

        max_stake = max(max_stake, stake)

        # ── Resolve ────────────────────────────────────────────────────
        won = (bet_on_a == team_a_wins[i])

        if won:
            net_profit      = stake * (bet_odds - 1.0) * (1.0 - commission)
            balance        += net_profit
            cumulative_loss = 0.0
            cur_win_streak += 1
            cur_loss_streak = 0
            max_win_streak  = max(max_win_streak, cur_win_streak)
        else:
            balance        -= stake
            cumulative_loss += stake
            cur_loss_streak += 1
            cur_win_streak   = 0
            max_loss_streak  = max(max_loss_streak, cur_loss_streak)

        peak_balance = max(peak_balance, balance)
        max_drawdown = max(max_drawdown, peak_balance - balance)

        steps_completed       += 1
        balance_hist[i + 1]    = balance

    return {
        'final_balance'  : balance,
        'max_drawdown'   : max_drawdown,
        'max_win_streak' : max_win_streak,
        'max_loss_streak': max_loss_streak,
        'max_stake'      : max_stake,
        'bust'           : bust,
        'bust_step'      : bust_step if bust else n_bets,
        'balance_history': balance_hist[: steps_completed + 1],
    }

## Single-run walkthrough
One representative simulation for each agent to get a feel for the dynamics.

In [ ]:
demo_rng = np.random.default_rng(seed=7)
fav_demo = simulate_martingale('favourite', local_rng=demo_rng)
demo_rng = np.random.default_rng(seed=7)   # same markets
out_demo = simulate_martingale('outsider',  local_rng=demo_rng)

fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)

for ax, res, label, color in [
    (axes[0], fav_demo, 'Favourites', FAVOURITE_COLOR),
    (axes[1], out_demo, 'Outsiders',  OUTSIDER_COLOR),
]:
    hist = res['balance_history']
    x    = np.arange(len(hist))
    ax.plot(x, hist, color=color, lw=0.8, alpha=0.9)
    ax.axhline(STARTING_BAL, color='grey', ls='--', lw=0.8, label='Starting balance')

    # Shade drawdown area
    running_peak = np.maximum.accumulate(hist)
    ax.fill_between(x, hist, running_peak, alpha=0.15, color='red', label='Drawdown')

    bust_tag = f"  ← BUST at step {res['bust_step']:,}" if res['bust'] else ''
    ax.set_title(
        f"{label} agent — "
        f"Max DD: ${res['max_drawdown']:,.0f}  "
        f"Max loss streak: {res['max_loss_streak']}  "
        f"Max stake: ${res['max_stake']:,.2f}  "
        f"Final: ${res['final_balance']:,.0f}{bust_tag}",
        fontsize=10,
    )
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.set_xlabel('Bet number')
    ax.set_ylabel('Balance')
    ax.legend(fontsize=8)

fig.suptitle('Single-run Martingale — Same Markets, Different Selection Strategy', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Monte Carlo — run 1,000 simulations per agent
Each simulation uses independent market draws.

In [ ]:
def run_monte_carlo(agent_type: str, n_sims: int = N_SIMS, seed: int = 0) -> pd.DataFrame:
    results = []
    for i in range(n_sims):
        sim_rng = np.random.default_rng(seed + i)
        r = simulate_martingale(agent_type, local_rng=sim_rng)
        results.append({
            'final_balance'  : r['final_balance'],
            'max_drawdown'   : r['max_drawdown'],
            'max_win_streak' : r['max_win_streak'],
            'max_loss_streak': r['max_loss_streak'],
            'max_stake'      : r['max_stake'],
            'bust'           : r['bust'],
            'bust_step'      : r['bust_step'],
        })
    return pd.DataFrame(results)

print(f'Running {N_SIMS:,} simulations per agent ({N_BETS:,} bets each)...')
fav_df = run_monte_carlo('favourite', seed=1000)
out_df = run_monte_carlo('outsider',  seed=2000)
print('Done.')

## Summary statistics

In [ ]:
def summary_table(df: pd.DataFrame, label: str) -> pd.DataFrame:
    bust_rate = df['bust'].mean() * 100
    survived  = df[~df['bust']]

    rows = {}
    for col, fmt, name in [
        ('final_balance',   '${:,.0f}',  'Final balance'),
        ('max_drawdown',    '${:,.0f}',  'Max drawdown'),
        ('max_stake',       '${:,.2f}',  'Max single stake'),
        ('max_loss_streak', '{:.1f}',    'Max loss streak'),
        ('max_win_streak',  '{:.1f}',    'Max win streak'),
    ]:
        rows[name] = {
            'Mean'  : fmt.format(df[col].mean()),
            'Median': fmt.format(df[col].median()),
            'Std'   : fmt.format(df[col].std()),
            'Min'   : fmt.format(df[col].min()),
            'Max'   : fmt.format(df[col].max()),
            'p95'   : fmt.format(df[col].quantile(0.95)),
        }

    rows['Bust rate'] = {
        'Mean': f'{bust_rate:.1f}%', 'Median': '—', 'Std': '—',
        'Min': '—', 'Max': '—', 'p95': '—',
    }
    if bust_rate > 0:
        rows['Avg bust step'] = {
            'Mean'  : f"{df[df['bust']]['bust_step'].mean():,.0f}",
            'Median': f"{df[df['bust']]['bust_step'].median():,.0f}",
            'Std'   : '—', 'Min': '—', 'Max': '—', 'p95': '—',
        }

    return pd.DataFrame(rows).T

print('=== FAVOURITES AGENT ===')
display(summary_table(fav_df, 'Favourites'))
print('\n=== OUTSIDERS AGENT ===')
display(summary_table(out_df, 'Outsiders'))

## Drawdown distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Max drawdown ──────────────────────────────────────────────────────────
ax = axes[0]
for df, label, color in [
    (fav_df, 'Favourites', FAVOURITE_COLOR),
    (out_df, 'Outsiders',  OUTSIDER_COLOR),
]:
    ax.hist(
        df['max_drawdown'],
        bins=60, alpha=0.55, color=color, label=label, density=True, edgecolor='none',
    )
    ax.axvline(df['max_drawdown'].median(), color=color, ls='--', lw=1.5,
               label=f'{label} median: ${df["max_drawdown"].median():,.0f}')

ax.set_title('Max Drawdown Distribution', fontweight='bold')
ax.set_xlabel('Max Drawdown ($)')
ax.set_ylabel('Density')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax.legend(fontsize=8)

# ── Max stake ────────────────────────────────────────────────────────────
ax = axes[1]
for df, label, color in [
    (fav_df, 'Favourites', FAVOURITE_COLOR),
    (out_df, 'Outsiders',  OUTSIDER_COLOR),
]:
    ax.hist(
        df['max_stake'],
        bins=60, alpha=0.55, color=color, label=label, density=True, edgecolor='none',
    )
    ax.axvline(df['max_stake'].median(), color=color, ls='--', lw=1.5,
               label=f'{label} median: ${df["max_stake"].median():,.2f}')

ax.set_title('Max Single Stake Required', fontweight='bold')
ax.set_xlabel('Max Stake ($)')
ax.set_ylabel('Density')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
ax.legend(fontsize=8)

fig.suptitle(f'Drawdown Risk — {N_SIMS:,} Simulations × {N_BETS:,} Bets', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Streak distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, title in [
    (axes[0], 'max_loss_streak', 'Max Consecutive Losses'),
    (axes[1], 'max_win_streak',  'Max Consecutive Wins'),
]:
    for df, label, color in [
        (fav_df, 'Favourites', FAVOURITE_COLOR),
        (out_df, 'Outsiders',  OUTSIDER_COLOR),
    ]:
        vals    = df[col]
        bins    = np.arange(vals.min(), vals.max() + 2) - 0.5
        ax.hist(
            vals, bins=bins, alpha=0.55, color=color, label=label,
            density=True, edgecolor='none',
        )
        ax.axvline(vals.median(), color=color, ls='--', lw=1.5,
                   label=f'{label} median: {vals.median():.0f}')

    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Streak length (bets)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)

fig.suptitle(f'Win / Loss Streak Distributions — {N_SIMS:,} Simulations', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Final balance distributions (survived runs only)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, label, color in [
    (axes[0], fav_df, 'Favourites', FAVOURITE_COLOR),
    (axes[1], out_df, 'Outsiders',  OUTSIDER_COLOR),
]:
    survived = df[~df['bust']]['final_balance']
    busted   = df[df['bust']]
    bust_pct = len(busted) / len(df) * 100

    ax.hist(survived, bins=50, color=color, alpha=0.7, edgecolor='none')
    ax.axvline(STARTING_BAL,       color='grey',  ls='--', lw=1.2, label='Starting balance')
    ax.axvline(survived.median(),  color='black', ls='-',  lw=1.5,
               label=f'Median: ${survived.median():,.0f}')

    ax.set_title(
        f'{label} — Survived runs ({100-bust_pct:.1f}%)\n'
        f'Bust rate: {bust_pct:.1f}%  |  Starting balance: ${STARTING_BAL:,.0f}',
        fontweight='bold', fontsize=9,
    )
    ax.set_xlabel('Final Balance ($)')
    ax.set_ylabel('Count')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.legend(fontsize=8)

fig.suptitle(f'Final Balance after {N_BETS:,} Bets — Survived Simulations Only', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

## Required bankroll to survive various percentiles

The Martingale system requires enough bankroll to cover the deepest losing run you'll encounter.
The table below shows the starting balance you'd need to survive the worst drawdown at each percentile.

In [ ]:
percentiles = [50, 75, 90, 95, 99]

rows = []
for p in percentiles:
    fav_dd = np.percentile(fav_df['max_drawdown'], p)
    out_dd = np.percentile(out_df['max_drawdown'], p)
    fav_sk = np.percentile(fav_df['max_stake'],    p)
    out_sk = np.percentile(out_df['max_stake'],    p)
    rows.append({
        'Percentile'                    : f'p{p}',
        'Fav — Max Drawdown'            : f'${fav_dd:>10,.0f}',
        'Out — Max Drawdown'            : f'${out_dd:>10,.0f}',
        'Fav — Max Single Stake'        : f'${fav_sk:>10,.2f}',
        'Out — Max Single Stake'        : f'${out_sk:>10,.2f}',
    })

req_df = pd.DataFrame(rows).set_index('Percentile')
display(req_df)

print(
    f"\nFav bust rate : {fav_df['bust'].mean()*100:.1f}%  "
    f"(of {N_SIMS} sims with ${STARTING_BAL:,.0f} starting bankroll)"
)
print(
    f"Out bust rate : {out_df['bust'].mean()*100:.1f}%  "
    f"(of {N_SIMS} sims with ${STARTING_BAL:,.0f} starting bankroll)"
)

## Head-to-head comparison chart

In [ ]:
fig = plt.figure(figsize=(14, 10))
gs  = GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

metrics = [
    ('max_drawdown',    'Max Drawdown ($)',        True),
    ('max_stake',       'Max Single Stake ($)',    True),
    ('max_loss_streak', 'Max Loss Streak (bets)',  False),
    ('max_win_streak',  'Max Win Streak (bets)',   False),
    ('final_balance',   'Final Balance ($)',        True),
]

for idx, (col, ylabel, dollar) in enumerate(metrics):
    row, c = divmod(idx, 3)
    ax = fig.add_subplot(gs[row, c])

    fav_vals = fav_df[col]
    out_vals = out_df[col]
    combined = np.concatenate([fav_vals, out_vals])
    bins = np.linspace(combined.min(), np.percentile(combined, 98), 50)

    ax.hist(fav_vals, bins=bins, alpha=0.55, color=FAVOURITE_COLOR,
            label='Favourites', density=True, edgecolor='none')
    ax.hist(out_vals, bins=bins, alpha=0.55, color=OUTSIDER_COLOR,
            label='Outsiders',  density=True, edgecolor='none')
    ax.axvline(fav_vals.median(), color=FAVOURITE_COLOR, ls='--', lw=1.5)
    ax.axvline(out_vals.median(), color=OUTSIDER_COLOR,  ls='--', lw=1.5)

    ax.set_title(ylabel, fontsize=9, fontweight='bold')
    ax.set_ylabel('Density', fontsize=8)
    if dollar:
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f'${v:,.0f}'))
    ax.tick_params(axis='x', labelsize=7)
    ax.legend(fontsize=7)

# 6th panel: bust rate bar chart
ax = fig.add_subplot(gs[1, 2])
bust_rates = [fav_df['bust'].mean() * 100, out_df['bust'].mean() * 100]
bars = ax.bar(['Favourites', 'Outsiders'], bust_rates,
              color=[FAVOURITE_COLOR, OUTSIDER_COLOR], alpha=0.75, width=0.5)
for bar, rate in zip(bars, bust_rates):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{rate:.1f}%', ha='center', va='bottom', fontweight='bold', fontsize=10)
ax.set_title('Bust Rate (%)', fontsize=9, fontweight='bold')
ax.set_ylabel('% of simulations', fontsize=8)
ax.set_ylim(0, max(bust_rates) * 1.3 + 2)

fig.suptitle(
    f'Martingale System — Full Comparison\n'
    f'{N_SIMS:,} simulations × {N_BETS:,} bets  |  '
    f'Starting bankroll: ${STARTING_BAL:,.0f}  |  '
    f'Overround: {OVERROUND*100:.0f}%  |  Commission: {COMMISSION*100:.0f}%',
    fontsize=11, fontweight='bold',
)
plt.show()